### JAB-Hessian sensitivity estimation & adaptive precision allocation
# Mistral-7B, **layer-streaming build for a free Colab T4**

Ported from the GPT-2 notebook, then restructured so it actually runs in Colab's free tier.

---

## Why the previous version crashed, and what changed

| | free Colab T4 | Mistral-7B fp16 |
|---|---|---|
| System RAM | **12.7 GB** | weights are **14.5 GB** |
| VRAM | **14.7 GiB** | 14.5 GB leaves ~0.2 GB for activations |
| Section 11 (joint FT) | — | wanted **two** copies = 29 GB |

`AutoModelForCausalLM.from_pretrained` stages the full 14.5 GB of weights in **system RAM**
before moving anything to the GPU. 14.5 > 12.7, so the process was OOM-killed *during loading* --
which is why it looked like "uses all available RAM and crashes" rather than a CUDA OOM. No
batch-size or sequence-length knob can fix that, because the crash happens before any of those
knobs are consulted.

**The model therefore can never be resident.** This notebook replaces the execution layer with a
layer-streaming pipeline -- the same structure AutoGPTQ and llm-awq use for large models:

1. **Weights are read one decoder layer at a time**, directly out of the mmap'd `safetensors`
   shards, straight onto the GPU (`CheckpointReader`). Peak host RAM is one tensor (~262 MB),
   not 14.5 GB. `MistralForCausalLM` is never instantiated.
2. **Hidden states are cached, not weights.** A pass embeds its inputs once, then walks the 32
   layers, propagating the cache through each layer and freeing that layer before loading the
   next. A calibration cache is `32 x 512 x 4096` fp16 = **134 MB**; the whole model's worth of
   activations is cheaper than 1% of its weights.
3. **The two-model requirement in section 11 is gone.** The original needed a float reference
   model for `target_A` alongside the model being quantized. Streaming gets the same semantics
   from **two hidden-state caches** -- `h_f` propagated through all-float layers, `h_q` through
   quantized ones -- plus a 100 MB fp32 snapshot of the current layer's float Q/K/V. Same
   objective, 29 GB -> ~0.25 GB.
4. **Quantization and evaluation are fused into one pass.** Since no quantized model is ever
   held, the evaluation windows ride along in a second hidden cache and are scored by
   `norm + lm_head` at the end of the pass.

Peak VRAM is now **~3 GB** and peak host RAM **under 1.5 GB**.

Two other T4-specific changes, both about the T4's *compute* rather than its memory:

* **GPTQ runs in fp32, not fp64.** The T4's fp64 throughput is 1/32 of fp32, and GPTQ's inner
  loop is 4096 sequential column updates per matrix. fp64 would cost ~4.5 minutes per `q_proj`,
  i.e. hours per pass. Hessians are still *accumulated* in fp64 (where precision actually
  matters); only the solve is fp32. Set `GPTQ_DTYPE = torch.float64` to revert.
* **The HAWQ-V2 perturbation term defaults to RTN on GPTQ's grid** (`PERTURB_MODE = "rtn"`).
  Measuring `||Q(W)-W||_F^2` with the full GPTQ solver at 5 bit-widths x 3 projections x 32
  layers is 480 GPTQ runs -- ~40 minutes of the T4's time to produce a *scoring heuristic*. RTN
  on the identical per-(channel, group) grid costs milliseconds and is the faithful reading of
  HAWQ-V2 anyway. `PERTURB_MODE = "gptq"` reproduces the original.

---

## Method (unchanged from the GPT-2 notebook)

0. Mistral architecture adapter (GQA shapes, RoPE, masks)
1. GPTQ core + quantization-grid helpers
2. Attention-aware joint loss `L = ||A(X)-A_hat(X)||^2 + lambda*KL(attention maps)`
3. Hutchinson trace estimator (double-backward HVPs)
4. Greedy sensitivity-per-cost bit allocator
5. **Layer-streaming engine** (new)
6. Calibration / evaluation data
7. **Validation**: the streaming forward vs. real `MistralForCausalLM`, exactly
8. The three pipeline passes
9. Setup -> 10. fp16 control -> 11. uniform 4-bit -> 12. JAB adaptive -> 13. joint FT -> 14. compare

The GPT-2 -> Mistral architecture differences (fused `c_attn` -> three `nn.Linear`s and the
`(d_out, d_in)` transpose that comes with it, GQA, RoPE, sliding window) are handled in
sections 0-2 and are validated numerically in section 7.

> **Runtime on a free T4:** ~5 min to download the checkpoint, then ~3-5 min per streaming pass.
> With all four passes enabled, budget ~30 minutes. Set `RUN_FP16_BASELINE = False` and
> `RUN_JOINT_FINETUNE = False` for a quicker first run.

In [ ]:
!pip install -q transformers datasets safetensors huggingface_hub sentencepiece protobuf

In [ ]:
import gc
import itertools
import json
import math
import os
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.autograd as autograd

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "mistralai/Mistral-7B-v0.1"

# Compute dtype for the one layer that is on the GPU at any moment. fp16, not bf16: the T4 is
# Turing (sm_75) and has no bf16 tensor cores.
GPU_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

# --- calibration / scoring ---
CALIB_N_SAMPLES   = 32      # calibration sequences
CALIB_SEQ_LEN     = 512
HESSIAN_N_BATCHES = 16      # batches used for H = 2 X^T X
HUTCH_SAMPLES     = 10      # Hutchinson probes per (layer, batch)
JAB_N_BATCHES     = 2       # batches averaged into each layer's trace
LAMBDA_KL         = 0.1     # 0.0 -> MSE-only ablation
GROUP_SIZE        = 128

# --- T4 compute choices (see the header) ---
GPTQ_DTYPE   = torch.float32
PERTURB_MODE = "rtn"        # "rtn" | "gptq"

# --- evaluation ---
# Perplexity is measured on the first N_EVAL_WINDOWS sliding windows of WikiText-2 test. Every
# configuration below uses the identical window set, so the comparison is exact; the absolute
# number is a subset perplexity and will sit slightly off a full-test figure.
N_EVAL_WINDOWS = 32
EVAL_MAX_LENGTH = 2048
EVAL_STRIDE     = 1024
LOGIT_CHUNK     = 256       # sequence positions per lm_head chunk (caps logit memory)

# --- which passes to run ---
RUN_FP16_BASELINE  = True   # unquantized control; also an end-to-end check on the streaming code
RUN_JOINT_FINETUNE = True

# --- joint fine-tuning ---
JOINT_STEPS_PER_BLOCK = 8
JOINT_LR              = 1e-4
JOINT_GRAD_CLIP       = 1.0
JOINT_LAYERS          = None   # e.g. range(0, 32, 4); others still get the GPTQ warm start


def free(*objs):
    """
    Run a collection and release cached VRAM.

    CAREFUL about what this does and does not do. The arguments are only there for readability:
    deleting them inside this function drops *this* frame's references, NOT the caller's bindings,
    so `free(x)` alone never releases `x`. For anything large -- a decoder layer, an fp32 weight
    snapshot, an optimizer -- the caller must `del` the name itself, otherwise the object survives
    until that name is rebound, and the next allocation happens while the old one is still live.
    That doubles the peak. The loops below `del` their layer explicitly for exactly this reason.
    """
    for o in objs:
        del o
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def vram(tag=""):
    if torch.cuda.is_available():
        a = torch.cuda.memory_allocated() / 1e9
        p = torch.cuda.max_memory_allocated() / 1e9
        print(f"    [vram{' ' + tag if tag else ''}: {a:.2f} GB now, {p:.2f} GB peak this pass]")


def reset_vram_peak():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


print("Device:", DEVICE)
if DEVICE == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}, {props.total_memory / 1e9:.1f} GB")
print("Model:", MODEL_ID, "| layer compute dtype:", GPU_DTYPE)

## 0. Mistral architecture adapter

Isolates every GPT-2 -> Mistral difference in one cell: GQA shape bookkeeping, the RoPE cache,
the causal + sliding-window mask, and RMSNorm.

`build_rope_cache` / `apply_rotary_pos_emb` replicate HF's implementation exactly --
`inv_freq = theta^(-2i/d)`, `emb = cat(freqs, freqs)`, and the `rotate_half` convention (which is
*not* the interleaved convention from the original RoPE paper). Section 7 pins this down against
the real model; that check is what would catch a future convention change upstream.

In [ ]:
class AttnConfig:
    """Shape/RoPE bookkeeping pulled out of a HF config, tolerant of 4.x vs 5.x naming."""

    def __init__(self, config):
        self.hidden_size   = config.hidden_size
        self.num_heads     = config.num_attention_heads
        self.num_kv_heads  = getattr(config, "num_key_value_heads", None) or config.num_attention_heads
        self.head_dim      = getattr(config, "head_dim", None) or config.hidden_size // config.num_attention_heads
        self.q_out         = self.num_heads * self.head_dim       # 32 * 128 = 4096
        self.kv_out        = self.num_kv_heads * self.head_dim    #  8 * 128 = 1024
        self.n_rep         = self.num_heads // self.num_kv_heads  # 4  (GQA expansion factor)
        self.scaling       = self.head_dim ** -0.5
        self.sliding_window = getattr(config, "sliding_window", None)
        self.rms_norm_eps  = getattr(config, "rms_norm_eps", 1e-6)
        self.n_layers      = config.num_hidden_layers

        # rope_theta moved into a `rope_parameters` dict in transformers 5.x
        rope_params = getattr(config, "rope_parameters", None)
        theta = rope_params.get("rope_theta") if isinstance(rope_params, dict) else None
        self.rope_theta = theta or getattr(config, "rope_theta", 10000.0)

        # Sizes of the three slices of the flattened [W_Q | W_K | W_V] vector.
        # NOTE: no longer three equal thirds -- that is the GQA consequence.
        self.q_numel  = self.hidden_size * self.q_out
        self.kv_numel = self.hidden_size * self.kv_out

    @property
    def qkv_numel(self):
        return self.q_numel + 2 * self.kv_numel

    def __repr__(self):
        return (f"AttnConfig(hidden={self.hidden_size}, layers={self.n_layers}, "
                f"heads={self.num_heads}, kv_heads={self.num_kv_heads}, "
                f"head_dim={self.head_dim}, n_rep={self.n_rep}, "
                f"rope_theta={self.rope_theta}, sliding_window={self.sliding_window})")


def rotate_half(x):
    """HF convention: split the head dim in half, not interleaved pairs."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]
    return torch.cat((-x2, x1), dim=-1)


def apply_rotary_pos_emb(q, k, cos, sin, unsqueeze_dim=1):
    cos = cos.unsqueeze(unsqueeze_dim)
    sin = sin.unsqueeze(unsqueeze_dim)
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)


def build_rope_cache(seq_len, cfg, device, dtype=torch.float32):
    """
    cos/sin of shape (1, seq_len, head_dim), matching MistralRotaryEmbedding for
    position_ids = arange(seq_len). Built in fp32 (HF forces fp32 here too), then cast.
    """
    inv_freq = 1.0 / (
        cfg.rope_theta
        ** (torch.arange(0, cfg.head_dim, 2, dtype=torch.int64, device=device).float() / cfg.head_dim)
    )
    pos = torch.arange(seq_len, device=device, dtype=torch.float32)
    freqs = torch.outer(pos, inv_freq)                  # (T, head_dim/2)
    emb = torch.cat((freqs, freqs), dim=-1)             # (T, head_dim)
    return emb.cos().to(dtype)[None], emb.sin().to(dtype)[None]


def build_attn_mask(seq_len, device, sliding_window=None):
    """
    Causal mask, intersected with Mistral's sliding window when one is configured.
    HF keeps position kv iff `kv <= q` and `kv > q - sliding_window`; for
    seq_len <= sliding_window (4096) this is exactly the plain causal mask.
    """
    mask = torch.ones(seq_len, seq_len, dtype=torch.bool, device=device).tril()
    if sliding_window is not None and seq_len > sliding_window:
        mask &= torch.ones(seq_len, seq_len, dtype=torch.bool, device=device).triu(-(sliding_window - 1))
    return mask


def repeat_kv(hidden_states, n_rep):
    """(B, n_kv_heads, T, head_dim) -> (B, n_kv_heads*n_rep, T, head_dim). GQA expansion."""
    if n_rep == 1:
        return hidden_states
    B, n_kv, T, d = hidden_states.shape
    return hidden_states[:, :, None, :, :].expand(B, n_kv, n_rep, T, d).reshape(B, n_kv * n_rep, T, d)


def rms_norm(x, weight, eps):
    """
    MistralRMSNorm as a function, for the final `model.norm` (whose module we never build).
    Cast order matters: HF normalizes in fp32, casts back, *then* multiplies by the weight.
    """
    dt = x.dtype
    h = x.to(torch.float32)
    h = h * torch.rsqrt(h.pow(2).mean(-1, keepdim=True) + eps)
    return weight * h.to(dt)

## 1. GPTQ core

`gptq_quantize_layer` is unchanged from the GPT-2 notebook -- it still works in the
`(d_in, d_out)` "x @ W" convention that GPT-2's Conv1D used. What is new:

* `gptq_quantize_linear`, which transposes `nn.Linear`'s `(d_out, d_in)` weight into that
  convention. **Getting this wrong is silent**: `q_proj` is square, so a missing `.T` runs
  without error and produces nonsense. Section 7 checks it.
* `hessian_from_hidden`, which accumulates `H = 2 X^T X` from a cached hidden state instead of a
  forward hook. Streaming has no full model to hook, and it does not need one: the input to
  `q_proj`/`k_proj`/`v_proj` *is* `input_layernorm(h)`, computed directly. One Hessian per layer
  serves all three projections, since they share that input.
* `grid_perturbation`, the HAWQ-V2 `||Q(W)-W||_F^2` term, on the identical per-(output channel,
  group) symmetric grid GPTQ uses -- either round-to-nearest (fast) or the full GPTQ solve.

In [ ]:
def _quantize_to_grid(w_col, scale, bits):
    """
    Symmetric per-output-row fake quantization of a single input-column (shape: d_out) using a
    fixed per-row scale computed up front from the original weight statistics.
    """
    qmax = 2 ** (bits - 1) - 1
    return torch.clamp(torch.round(w_col / scale), -qmax, qmax) * scale


@torch.no_grad()
def gptq_quantize_layer(weight_in_out, H, bits=4, damp_percent=0.01, group_size=None,
                        act_order=True, return_scale=False, work_dtype=None):
    """
    Quantizes a weight matrix in the (d_in, d_out) "x @ W" convention using GPTQ.
    UNCHANGED from the GPT-2 notebook apart from the `work_dtype` knob.

    weight_in_out: (d_in, d_out) float tensor
    H: (d_in, d_in) Hessian
    damp_percent: Hessian damping for numerical stability (GPTQ default ~0.01)
    group_size: separate per-row scale per contiguous group of `group_size` input columns
    act_order: quantize columns in order of decreasing Hessian diagonal
    return_scale: ALSO return the exact per-(output channel, group) scale used, in the ORIGINAL
        (unpermuted) column order and the same (d_in, d_out) orientation -- so a caller can later
        fake-quantize the SAME weight onto the SAME grid (needed for the STE warm start).
    work_dtype: fp64 is GPTQ-canonical; fp32 is ~30x faster on a T4 (see the header).

    Returns the fake-quantized weight (also written into weight_in_out in place), or
    (W_final, scale) if return_scale.
    """
    work_dtype = work_dtype or GPTQ_DTYPE
    device = weight_in_out.device
    W = weight_in_out.detach().clone().to(work_dtype).T.contiguous()  # (d_out, d_in)
    d_out, d_in = W.shape

    H = H.clone().to(work_dtype)
    mean_diag = H.diagonal().mean()
    H += damp_percent * mean_diag * torch.eye(d_in, dtype=work_dtype, device=device)

    if act_order:
        perm = torch.argsort(torch.diag(H), descending=True)
        invperm = torch.argsort(perm)
        W = W[:, perm]
        H = H[perm][:, perm]
    else:
        perm = invperm = torch.arange(d_in, device=device)

    H_inv = torch.linalg.inv(H)
    free(H)

    # Per-(row, group) scale, from the original weights in the permuted column order, before any
    # quantization.
    qmax = 2 ** (bits - 1) - 1
    gs = group_size if group_size is not None else d_in
    n_groups = (d_in + gs - 1) // gs
    scale = torch.zeros(d_out, n_groups, dtype=work_dtype, device=device)
    for g in range(n_groups):
        start, end = g * gs, min((g + 1) * gs, d_in)
        scale[:, g] = (W[:, start:end].abs().amax(dim=1) / qmax).clamp(min=1e-8)

    for i in range(d_in):
        row_scale = scale[:, i // gs]
        w_col = W[:, i]
        q_col = _quantize_to_grid(w_col, row_scale, bits)
        err = (w_col - q_col) / H_inv[i, i]
        if i + 1 < d_in:
            W[:, i + 1:] -= torch.outer(err, H_inv[i, i + 1:])
        W[:, i] = q_col
    free(H_inv)

    if act_order:
        W = W[:, invperm]

    W_final = W.T.contiguous().to(weight_in_out.dtype)  # back to (d_in, d_out)
    weight_in_out.copy_(W_final)

    if not return_scale:
        return W_final

    group_idx = torch.arange(d_in, device=device) // gs
    scale_cols = scale[:, group_idx]                                     # (d_out, d_in), permuted
    scale_cols = scale_cols[:, invperm] if act_order else scale_cols
    return W_final, scale_cols.T.contiguous().to(weight_in_out.dtype)


@torch.no_grad()
def gptq_quantize_linear(linear, H, bits=4, damp_percent=0.01, group_size=None,
                         act_order=True, return_scale=False):
    """
    GPTQ for an `nn.Linear`, whose weight is (d_out, d_in) -- the TRANSPOSE of what
    gptq_quantize_layer expects. Quantizes in place in the module's storage dtype, and returns
    GPTQ's fp32 output so callers needing a bit-exact warm start are not limited to fp16
    precision.

    Returns W_io (d_in, d_out) [, scale (d_in, d_out)].
    """
    W_io = linear.weight.data.T.contiguous().to(torch.float32)   # (d_in, d_out)
    out = gptq_quantize_layer(W_io, H, bits=bits, damp_percent=damp_percent,
                              group_size=group_size, act_order=act_order,
                              return_scale=return_scale)
    W_q, scale = out if return_scale else (out, None)
    linear.weight.data.copy_(W_q.T.to(linear.weight.dtype))
    return (W_q, scale) if return_scale else W_q


@torch.no_grad()
def quantize_qkv(projs, H, bits, group_size=None, damp_percent=0.01, return_scales=False):
    """
    Applies GPTQ to (q_proj, k_proj, v_proj) with the shared Hessian H. The Mistral equivalent of
    one `gptq_quantize_layer(c_attn.weight.data, ...)` call on GPT-2's fused matrix.
    """
    group_size = GROUP_SIZE if group_size is None else group_size
    ws, scales = [], []
    for proj in projs:
        out = gptq_quantize_linear(proj, H, bits=bits, damp_percent=damp_percent,
                                   group_size=group_size, act_order=True,
                                   return_scale=return_scales)
        if return_scales:
            ws.append(out[0]); scales.append(out[1])
        else:
            ws.append(out)
    return (ws, scales) if return_scales else ws


@torch.no_grad()
def hessian_from_hidden(layer, hiddens, hidden_size, device):
    """
    H = 2 X^T X, averaged over tokens, where X = input_layernorm(h) is exactly the tensor the
    GPT-2 notebook captured with a forward pre-hook on `c_attn`.

    Accumulated in fp64 -- accumulation precision is what matters for a 4096x4096 Gram matrix
    summed over tens of thousands of tokens, even when the downstream solve runs in fp32.
    """
    H = torch.zeros(hidden_size, hidden_size, dtype=torch.float64, device=device)
    n = 0
    for h in hiddens:
        X = layer.input_layernorm(h).reshape(-1, hidden_size).to(torch.float64)
        H.addmm_(X.T, X, alpha=2.0)
        n += X.shape[0]
        free(X)
    if n:
        H /= n
    return H


@torch.no_grad()
def grid_perturbation(W_io, bits, group_size=None, H=None, mode=None):
    """
    HAWQ-V2's `||Q(W) - W||_F^2`, measured in weight space on GPTQ's own per-(output channel,
    group) symmetric grid.

    mode="rtn"  : round-to-nearest on that grid. Milliseconds. The faithful reading of HAWQ-V2,
                  whose Omega uses the quantizer's perturbation, not a solver's residual.
    mode="gptq" : the full GPTQ solve (requires H). Reproduces the GPT-2 notebook exactly, at
                  ~480 solver runs per scoring pass -- see the header for why that is not a T4
                  default.
    """
    mode = PERTURB_MODE if mode is None else mode
    if mode == "gptq":
        if H is None:
            raise ValueError('mode="gptq" needs the Hessian H')
        W_q = gptq_quantize_layer(W_io.clone(), H, bits=bits, group_size=group_size, act_order=True)
        out = (W_q - W_io).pow(2).sum().item()
        free(W_q)
        return out

    W = W_io.T                                   # (d_out, d_in)
    d_out, d_in = W.shape
    gs = group_size if group_size is not None else d_in
    qmax = 2 ** (bits - 1) - 1
    total = 0.0
    for start in range(0, d_in, gs):
        blk = W[:, start:min(start + gs, d_in)].to(torch.float32)
        scale = (blk.abs().amax(dim=1, keepdim=True) / qmax).clamp(min=1e-8)
        blk_q = torch.clamp(torch.round(blk / scale), -qmax, qmax) * scale
        total += (blk_q - blk).pow(2).sum().item()
    return total

## 2. Attention-aware joint loss
`L = ||A(X) - A_hat(X)||^2 (+ lambda * KL(attention maps))` -- a from-scratch, differentiable
multi-head attention as a pure function of a flattened `[W_Q | W_K | W_V]` vector, which is what
the Hutchinson estimator differentiates through.

**The three Mistral-specific changes here are the heart of the port:**
1. `reshape_weights` splits by explicit `(q_numel, kv_numel)` sizes rather than equal thirds,
   because GQA makes `W_K`/`W_V` a quarter the width of `W_Q`.
2. `compute_attention` applies **RoPE** to Q and K and **`repeat_kv`** to K and V. Without RoPE
   this computes a different operator than the real model, so the Hessian trace would describe a
   model that does not exist.
3. Softmax in fp32 and a causal mask intersected with the sliding window, matching HF's
   `eager_attention_forward`.

In the streaming build this function does double duty: it is both the differentiable loss target
**and** the attention step of the actual forward pass (section 5), so the propagated hidden states
and the fine-tuning objective are computed by the identical code path. Bias arguments are kept
for continuity with the GPT-2 version; Mistral builds all projections with `bias=False`.

In [ ]:
def reshape_weights(w_flat, cfg):
    """
    Flat vector -> Q, K, V in the (d_in, d_out) "x @ W" convention.

    GPT-2 split into three equal (n_embd, n_embd) blocks. Mistral-7B has 32 query heads but only
    8 KV heads, so:  W_Q: (4096, 4096),  W_K: (4096, 1024),  W_V: (4096, 1024).
    """
    h, q_n, kv_n = cfg.hidden_size, cfg.q_numel, cfg.kv_numel
    return (w_flat[0:q_n].reshape(h, cfg.q_out),
            w_flat[q_n:q_n + kv_n].reshape(h, cfg.kv_out),
            w_flat[q_n + kv_n:q_n + 2 * kv_n].reshape(h, cfg.kv_out))


def flatten_weights(W_Q, W_K, W_V):
    """Inverse of reshape_weights (all three in (d_in, d_out) orientation)."""
    return torch.cat([W_Q.reshape(-1), W_K.reshape(-1), W_V.reshape(-1)])


def compute_attention(W_Q, W_K, W_V, X, cfg, b_Q=None, b_K=None, b_V=None,
                      cos=None, sin=None, attn_mask=None):
    """
    Attention output and attention weights, matching HF's MistralAttention exactly.

    W_Q: (hidden, num_heads*head_dim); W_K/W_V: (hidden, num_kv_heads*head_dim)
    X: (batch, seq_len, hidden)
    cos/sin: RoPE cache; built on the fly if omitted (pass them in to avoid rebuilding inside
        every Hutchinson probe and every layer of a streaming pass)

    Returns A_hat (batch, seq_len, num_heads*head_dim) -- heads merged, pre-o_proj -- and
    attn_weights (batch, num_heads, seq_len, seq_len), post-softmax and post-GQA-expansion.
    """
    B, T, _ = X.shape

    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V
    if b_Q is not None:                      # Mistral is bias-free; kept for compatibility
        Q, K, V = Q + b_Q, K + b_K, V + b_V

    Q = Q.view(B, T, cfg.num_heads,    cfg.head_dim).transpose(1, 2)   # (B, 32, T, 128)
    K = K.view(B, T, cfg.num_kv_heads, cfg.head_dim).transpose(1, 2)   # (B,  8, T, 128)
    V = V.view(B, T, cfg.num_kv_heads, cfg.head_dim).transpose(1, 2)

    # RoPE on Q and K
    if cos is None or sin is None:
        cos, sin = build_rope_cache(T, cfg, X.device, dtype=X.dtype)
    Q, K = apply_rotary_pos_emb(Q, K, cos, sin)

    # GQA: expand 8 KV heads to 32 to match the query heads
    K = repeat_kv(K, cfg.n_rep)
    V = repeat_kv(V, cfg.n_rep)

    scores = (Q @ K.transpose(-2, -1)) * cfg.scaling

    if attn_mask is None:
        attn_mask = build_attn_mask(T, X.device, cfg.sliding_window)
    scores = scores.masked_fill(~attn_mask, float("-inf"))

    # HF takes the softmax in fp32 then casts back; mirror that for numerical parity.
    attn_weights = torch.softmax(scores, dim=-1, dtype=torch.float32).to(Q.dtype)

    A_hat = (attn_weights @ V).transpose(1, 2).contiguous().view(B, T, cfg.q_out)
    return A_hat, attn_weights


def mse_loss(w_flat, X, target_A, cfg, b_Q=None, b_K=None, b_V=None, cos=None, sin=None,
             attn_mask=None):
    """L_mse = ||A(X) - A_hat(X)||^2 -- the primary loss."""
    W_Q, W_K, W_V = reshape_weights(w_flat, cfg)
    A_hat, _ = compute_attention(W_Q, W_K, W_V, X, cfg, b_Q=b_Q, b_K=b_K, b_V=b_V,
                                 cos=cos, sin=sin, attn_mask=attn_mask)
    return F.mse_loss(A_hat, target_A)


def kl_loss(w_flat, X, target_attn, cfg, b_Q=None, b_K=None, b_V=None, cos=None, sin=None,
            attn_mask=None):
    """L_kl = KL(target attention maps || predicted attention maps)."""
    W_Q, W_K, W_V = reshape_weights(w_flat, cfg)
    _, attn_weights = compute_attention(W_Q, W_K, W_V, X, cfg, b_Q=b_Q, b_K=b_K, b_V=b_V,
                                        cos=cos, sin=sin, attn_mask=attn_mask)
    # KL(P || Q) = sum(P * log(P/Q)); P = target_attn, Q = attn_weights.
    # Masked entries have P = 0 and contribute 0 (F.kl_div uses xlogy).
    log_q = torch.log(attn_weights + 1e-8)  # epsilon for stability
    return F.kl_div(log_q, target_attn, reduction="batchmean")  # as in Q-BERT & APTQ


def attention_loss(w_flat, X, target_A, cfg, target_attn=None, lambda_kl=0.1,
                   b_Q=None, b_K=None, b_V=None, cos=None, sin=None, attn_mask=None):
    """MSE only if target_attn is None or lambda_kl == 0; otherwise MSE + lambda_kl * KL."""
    loss = mse_loss(w_flat, X, target_A, cfg, b_Q=b_Q, b_K=b_K, b_V=b_V,
                    cos=cos, sin=sin, attn_mask=attn_mask)
    if target_attn is not None and lambda_kl:
        loss = loss + lambda_kl * kl_loss(w_flat, X, target_attn, cfg, b_Q=b_Q, b_K=b_K, b_V=b_V,
                                          cos=cos, sin=sin, attn_mask=attn_mask)
    return loss

## 3. Hutchinson trace estimator
`trace(H) ~= (1/n) * sum(v_i^T H v_i)` for Rademacher `v_i`, via double-backward HVPs -- the full
Hessian is never formed.

Unchanged. Note the scale it now runs at: `params` is a 25.2M-element vector
(`4096*4096 + 2*4096*1024`) rather than GPT-2's 1.8M, so each probe's graph is ~14x larger. That
is why `HUTCH_SAMPLES` defaults to 10 rather than HAWQ-V2's 50; raise it if you have budget,
since variance reduction is exactly what it buys.

In [ ]:
def hessian_vector_product(loss_fn, params, vector, retain_graph=True):
    # create_graph=True keeps the graph alive for the second grad -- do not release it
    grad = autograd.grad(loss_fn(params), params, create_graph=True, retain_graph=True)[0]
    hvp = autograd.grad(grad, params, grad_outputs=vector, retain_graph=True)[0]
    del grad
    return hvp


def hutchinson_trace_estimator(loss_fn, params, samples=50):
    # trace(H) ~= (1/n) * sum(v_i^T H v_i)
    if not params.requires_grad:
        params.requires_grad_(True)

    estimated_trace = 0.0
    for _ in range(samples):
        # Rademacher vector (as in HAWQ-V2)
        vec = (torch.randint(0, 2, params.shape, device=params.device) * 2 - 1).to(params.dtype)
        hvp = hessian_vector_product(loss_fn, params, vec)
        estimated_trace += torch.dot(vec.flatten(), hvp.flatten())
        del vec, hvp
    return estimated_trace / samples

## 4. Greedy sensitivity-per-cost allocator
Starts every layer at the highest bit-width, repeatedly downgrades whichever layer loses the
least accuracy per unit of budget freed until the budget is met, then spends any leftover on the
best available upgrades.

Algorithmically unchanged. Mistral consequences: 32 units instead of 12 (so
`budget = target_avg_bits * 32`), and `brute_force_optimal` is now `5^32` combinations -- kept for
reference behind a guard.

In [ ]:
BIT_WIDTHS = [2, 3, 4, 8, 16]


def cost(bits, param_count=1.0):
    """
    Memory cost of one unit at a given bit-width. Every Mistral-7B layer holds the same number of
    Q/K/V parameters (25.17M), so param_count=1.0 keeps `budget` directly readable as
    "average bits per layer", exactly as in the GPT-2 version.
    """
    return bits * param_count


def greedy_allocate(scores, budget):
    """
    scores: {unit_name: {bits: sensitivity}};  budget: max total cost.
      1. Start every unit at the HIGHEST bit-width.
      2. While over budget, apply the downgrade with the best cost-saved-per-accuracy-lost ratio.
      3. Spend leftover budget on the best affordable upgrades.
    """
    current_bits = {name: max(BIT_WIDTHS) for name in scores}

    def total_cost():
        return sum(cost(current_bits[n]) for n in scores)

    def total_sensitivity():
        return sum(scores[n][current_bits[n]] for n in scores)

    while total_cost() > budget:                     # downgrade loop
        best = best_bits = best_ratio = None
        for name in scores:
            lower = [b for b in BIT_WIDTHS if b < current_bits[name]]
            if not lower:
                continue                             # already at the lowest bit-width
            nb = max(lower)
            saved = cost(current_bits[name]) - cost(nb)
            added = scores[name][nb] - scores[name][current_bits[name]]
            ratio = added / saved
            if best_ratio is None or ratio < best_ratio:
                best_ratio, best, best_bits = ratio, name, nb
        if best is None:
            break                                    # nothing left to downgrade
        current_bits[best] = best_bits

    made_an_upgrade = True                           # spend any remaining budget
    while made_an_upgrade:
        made_an_upgrade = False
        best = best_bits = best_ratio = None
        for name in scores:
            higher = [b for b in BIT_WIDTHS if b > current_bits[name]]
            if not higher:
                continue
            nb = min(higher)
            extra = cost(nb) - cost(current_bits[name])
            if total_cost() + extra > budget:
                continue                             # can't afford it
            ratio = (scores[name][current_bits[name]] - scores[name][nb]) / extra
            if best_ratio is None or ratio > best_ratio:
                best_ratio, best, best_bits = ratio, name, nb
        if best is not None:
            current_bits[best] = best_bits
            made_an_upgrade = True

    return current_bits, total_cost(), total_sensitivity()


def brute_force_optimal(scores, budget, max_units=8):
    """
    Exhaustive search, for sanity-checking the greedy allocator on a small subset.
    Guarded: with Mistral's 32 layers this is 5^32 ~ 2.3e22 combinations.
    """
    names = list(scores.keys())
    if len(names) > max_units:
        raise ValueError(f"brute_force_optimal over {len(names)} units = "
                         f"{len(BIT_WIDTHS)}^{len(names)} combinations. Restrict `scores` to at "
                         f"most {max_units} units, or raise max_units if you really mean it.")
    best_assignment = best_cost = None
    best_sensitivity = float("inf")
    for combo in itertools.product(*([BIT_WIDTHS] * len(names))):
        c = sum(cost(b) for b in combo)
        if c > budget:
            continue
        s = sum(scores[names[i]][combo[i]] for i in range(len(names)))
        if s < best_sensitivity:
            best_sensitivity, best_cost = s, c
            best_assignment = dict(zip(names, combo))
    return best_assignment, best_cost, best_sensitivity


def scores_to_allocator_format(jab_scores, bit_widths=None):
    """Heuristic table (trace / bits^1.5), kept as an alternative to the HAWQ-V2 measured form."""
    bit_widths = BIT_WIDTHS if bit_widths is None else bit_widths
    return {n: {b: t / (b ** 1.5) for b in bit_widths} for n, t in jab_scores.items()}


def layer_name(i):
    return f"layer_{i}_QKV"


def layer_idx_of(name):
    return int(name.split("_")[1])

## 5. Layer-streaming engine

This is the cell that makes the notebook fit on a T4. Three pieces:

**`CheckpointReader`** downloads the `safetensors` shards once and hands out individual tensors
on demand, straight to the GPU. Peak host RAM is one tensor at a time (262 MB for
`embed_tokens`), never the 14.5 GB whole. `MistralForCausalLM` is never constructed.

**`load_decoder_layer`** builds one `MistralDecoderLayer` on the `meta` device (allocating
nothing) and adopts the checkpoint tensors into it with `load_state_dict(..., assign=True)`. One
layer is ~436 MB in fp16. Note the dtype hop: the checkpoint is bf16, but the T4 has no bf16
tensor cores, so each layer is converted to fp16 *as it is read* -- converting the whole
checkpoint up front is what would blow out host RAM.

**`decoder_layer_forward`** runs a layer manually rather than calling `layer.forward`. Two
reasons: it sidesteps the `position_embeddings` / `past_key_values` / return-type churn across
transformers versions, and it routes attention through section 2's `compute_attention`, so the
propagated hidden states and the fine-tuning objective come from **the same validated code
path**. It also yields the attention maps for the KL target for free, with no
`output_attentions` plumbing. The `w_qkv` override lets a caller run the layer with *different*
Q/K/V than the module holds -- which is how float targets are computed after the module's own
weights have been quantized.

In [ ]:
from huggingface_hub import snapshot_download
from safetensors import safe_open
from transformers import AutoConfig, AutoTokenizer
from transformers.models.mistral.modeling_mistral import MistralDecoderLayer

LAYER_KEYS = ["self_attn.q_proj.weight", "self_attn.k_proj.weight", "self_attn.v_proj.weight",
              "self_attn.o_proj.weight", "mlp.gate_proj.weight", "mlp.up_proj.weight",
              "mlp.down_proj.weight", "input_layernorm.weight", "post_attention_layernorm.weight"]


class CheckpointReader:
    """
    Random access to a sharded safetensors checkpoint, one tensor at a time.

    The whole point: `from_pretrained` materializes every parameter in host RAM before any of it
    reaches the GPU, which is what exceeded Colab's 12.7 GB. This never holds more than the single
    tensor being read.
    """

    def __init__(self, model_id, device):
        self.device = str(device)
        self.local = snapshot_download(model_id, allow_patterns=[
            "*.safetensors", "*.safetensors.index.json", "config.json",
            "generation_config.json", "tokenizer*", "*.model", "special_tokens_map.json",
        ])
        index = os.path.join(self.local, "model.safetensors.index.json")
        if os.path.exists(index):
            with open(index, encoding="utf-8") as f:
                self.weight_map = json.load(f)["weight_map"]
        else:                                       # single-shard checkpoint
            fname = "model.safetensors"
            with safe_open(os.path.join(self.local, fname), framework="pt") as h:
                self.weight_map = {k: fname for k in h.keys()}
        self._handles = {}
        # safetensors can read straight to CUDA, skipping host memory entirely. Not every build
        # supports it, so probe with a real open AND a real read, then fall back. safe_open wants
        # an indexed device ("cuda:0"), not the bare "cuda" torch accepts.
        probe_name, probe_file = next(iter(self.weight_map.items()))
        self._read_device = "cuda:0" if self.device.startswith("cuda") else self.device
        try:
            self._handle(probe_file).get_tensor(probe_name)
        except Exception as e:
            self._handles.clear()
            self._read_device = "cpu"
            print(f"  [safetensors direct-to-GPU read unavailable ({type(e).__name__}); staging "
                  f"one tensor at a time through host RAM instead -- still bounded, just slower]")

    def _handle(self, fname):
        if fname not in self._handles:
            self._handles[fname] = safe_open(os.path.join(self.local, fname),
                                             framework="pt", device=self._read_device)
        return self._handles[fname]

    def has(self, name):
        return name in self.weight_map

    def get(self, name, dtype=None, device=None):
        t = self._handle(self.weight_map[name]).get_tensor(name)
        return t.to(device or self.device, dtype or GPU_DTYPE)

    def close(self):
        self._handles.clear()


@torch.no_grad()
def load_decoder_layer(reader, config, idx, device=None, dtype=None):
    """
    One decoder layer, on the GPU, in `dtype` -- allocating nothing else.

    Built on the `meta` device so no host storage is ever created, then the checkpoint tensors are
    adopted with assign=True (no copy into pre-allocated storage).
    """
    device = device or DEVICE
    dtype = dtype or GPU_DTYPE
    with torch.device("meta"):
        layer = MistralDecoderLayer(config, layer_idx=idx)
    sd = {k: reader.get(f"model.layers.{idx}.{k}", dtype=dtype, device=device) for k in LAYER_KEYS}
    layer.load_state_dict(sd, strict=True, assign=True)
    layer.eval()
    for p in layer.parameters():
        p.requires_grad_(False)
    return layer


def qkv_of(layer):
    a = layer.self_attn
    return a.q_proj, a.k_proj, a.v_proj


def qkv_weights_io(layer, dtype=None):
    """The layer's Q/K/V in the (d_in, d_out) convention -- i.e. transposed out of nn.Linear."""
    return tuple(p.weight.data.T.to(dtype) if dtype else p.weight.data.T for p in qkv_of(layer))


@torch.no_grad()
def write_qkv_flat(layer, w_flat, cfg):
    """Scatter a flat [W_Q|W_K|W_V] vector back into the layer's three projections."""
    for proj, W in zip(qkv_of(layer), reshape_weights(w_flat, cfg)):
        proj.weight.data.copy_(W.T.to(proj.weight.dtype))


def decoder_layer_forward(layer, h, cfg, cos, sin, attn_mask, w_qkv=None, return_attn=False):
    """
    One decoder layer, run manually: RMSNorm -> attention (section 2) -> o_proj -> residual ->
    RMSNorm -> MLP -> residual.

    w_qkv: optional (W_Q, W_K, W_V) in (d_in, d_out) form, used INSTEAD of the module's own
        projections. This is how float targets are produced after the module has been quantized.
    """
    x = layer.input_layernorm(h)
    W_Q, W_K, W_V = w_qkv if w_qkv is not None else qkv_weights_io(layer)
    A, attn_w = compute_attention(W_Q, W_K, W_V, x, cfg, cos=cos, sin=sin, attn_mask=attn_mask)
    h = h + layer.self_attn.o_proj(A.to(layer.self_attn.o_proj.weight.dtype))
    h = h + layer.mlp(layer.post_attention_layernorm(h))
    return (h, attn_w) if return_attn else (h, None)


@torch.no_grad()
def embed_cache(reader, ids_list, device=None, dtype=None):
    """Hidden states entering layer 0, for every sequence. embed_tokens is freed immediately."""
    device = device or DEVICE
    dtype = dtype or GPU_DTYPE
    emb = reader.get("model.embed_tokens.weight", dtype=dtype, device=device)
    out = [F.embedding(ids.to(device), emb) for ids in ids_list]
    free(emb)
    return out


@torch.no_grad()
def score_perplexity(reader, hiddens, windows, cfg, device=None, chunk=None):
    """
    Final RMSNorm + lm_head over the propagated evaluation hiddens -> WikiText-2 perplexity.

    Logits are produced in `chunk`-position slices so the (T x 32000) fp32 tensor never
    materializes in full. `windows` supplies (input_ids, trg_len) per hidden state; only the last
    `trg_len` positions of each window are scored, which is the standard sliding-window scheme.
    """
    device = device or DEVICE
    chunk = chunk or LOGIT_CHUNK
    norm_w = reader.get("model.norm.weight", dtype=GPU_DTYPE, device=device)
    head_key = "lm_head.weight" if reader.has("lm_head.weight") else "model.embed_tokens.weight"
    head_w = reader.get(head_key, dtype=GPU_DTYPE, device=device)     # (vocab, hidden)

    nll_sum, n_tokens = 0.0, 0
    for h, (ids, trg_len) in zip(hiddens, windows):
        hh = rms_norm(h, norm_w, cfg.rms_norm_eps)
        labels = ids.to(device).clone()
        if trg_len < labels.shape[1]:
            labels[:, :-trg_len] = -100
        T = hh.shape[1]
        for s in range(0, T - 1, chunk):
            e = min(s + chunk, T - 1)
            logits = (hh[:, s:e] @ head_w.T).float()                  # (1, L, vocab)
            tgt = labels[:, s + 1:e + 1]
            nll_sum += F.cross_entropy(logits.reshape(-1, logits.shape[-1]), tgt.reshape(-1),
                                       ignore_index=-100, reduction="sum").item()
            n_tokens += int((tgt != -100).sum())
            free(logits)
        free(hh)
    free(norm_w, head_w)
    return math.exp(nll_sum / n_tokens)

## 6. Calibration and evaluation data

Calibration chunks come from WikiText-2 train, evaluation windows from WikiText-2 test with the
usual `max_length`/`stride` sliding scheme.

`N_EVAL_WINDOWS` truncates the test set. This is a real limitation and worth stating plainly:
with 32 windows the reported figure is a **subset perplexity** over ~33k scored tokens, not a
full-test number, so it will not line up with published Mistral-7B WikiText-2 values. Every
configuration in this notebook is scored on the identical window set, so the *comparisons* --
which is what the experiment is about -- are exact. Raising `N_EVAL_WINDOWS` costs
`2048 x 4096 x 2 B = 16 MB` of VRAM per window plus time.

In [ ]:
from datasets import load_dataset


def build_calibration_ids(tokenizer, n_samples=None, seq_len=None):
    """`n_samples` chunks of `seq_len` tokens from WikiText-2 train, as (1, seq_len) id tensors."""
    n_samples = CALIB_N_SAMPLES if n_samples is None else n_samples
    seq_len = CALIB_SEQ_LEN if seq_len is None else seq_len

    raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
    text = "\n\n".join(t for t in raw["text"] if t.strip())
    # add_special_tokens=False: Mistral's tokenizer prepends <s>, which we do not want spliced
    # into the middle of a mid-corpus chunk.
    ids = tokenizer(text, return_tensors="pt", add_special_tokens=False).input_ids[0]

    out = []
    for i in range(n_samples):
        start = i * seq_len
        if start + seq_len > ids.shape[0]:
            break
        out.append(ids[start:start + seq_len].unsqueeze(0))
    return out


def build_eval_windows(tokenizer, n_windows=None, max_length=None, stride=None):
    """
    Sliding windows over WikiText-2 test as [(input_ids, trg_len)], matching the scheme the GPT-2
    notebook's evaluate_perplexity used: window w scores only the positions not already scored by
    window w-1.
    """
    n_windows = N_EVAL_WINDOWS if n_windows is None else n_windows
    max_length = EVAL_MAX_LENGTH if max_length is None else max_length
    stride = EVAL_STRIDE if stride is None else stride

    raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
    text = "\n\n".join(t for t in raw["text"] if t.strip())
    ids = tokenizer(text, return_tensors="pt").input_ids[0]
    seq_len = ids.shape[0]

    windows, prev_end = [], 0
    for begin in range(0, seq_len, stride):
        end = min(begin + max_length, seq_len)
        if end - begin < 2:
            break
        windows.append((ids[begin:end].unsqueeze(0), end - prev_end))
        prev_end = end
        if end == seq_len or len(windows) >= n_windows:
            break
    return windows

## 7. Validation

Two checks, both cheap, and both worth running before trusting any number below.

**7a** builds a *tiny random* Mistral (real `MistralForCausalLM`, real GQA, real RoPE, and a
`sliding_window` deliberately smaller than the sequence length so that code path is genuinely
exercised) and compares its own `forward` against the streaming path -- `embed_cache` ->
`decoder_layer_forward` x N -> `rms_norm` + head. This pins down, in one assertion, the weight
transpose, the GQA expansion, the RoPE convention, the mask convention, the residual structure
and the RMSNorm cast order. It runs in about a second and needs no downloads.

**7b** is the control that catches anything 7a cannot: the *unquantized* fp16 streaming
perplexity on the real Mistral-7B (section 10). If the streaming forward were subtly wrong, that
number would be nonsense rather than the ~5-6 range Mistral-7B produces on WikiText-2.

In [ ]:
def _validate_streaming_against_hf(verbose=True):
    """7a: streaming forward vs. a real MistralForCausalLM, on a tiny random model."""
    from transformers import MistralConfig, MistralForCausalLM

    torch.manual_seed(0)
    tiny = MistralConfig(
        vocab_size=256, hidden_size=64, intermediate_size=128, num_hidden_layers=3,
        num_attention_heads=8, num_key_value_heads=2,      # GQA, n_rep = 4
        max_position_embeddings=256, sliding_window=8,     # smaller than T below, on purpose
        rms_norm_eps=1e-6,
    )
    ref = MistralForCausalLM(tiny)
    ref.config._attn_implementation = "eager"
    ref.eval().to(torch.float32)

    cfg = AttnConfig(ref.config)
    assert (cfg.num_heads, cfg.num_kv_heads, cfg.head_dim, cfg.n_rep) == (8, 2, 8, 4)
    assert cfg.q_out == 64 and cfg.kv_out == 16          # GQA: K/V are a quarter of Q

    T = 24                                                # > sliding_window, so the window bites
    ids = torch.randint(0, 256, (1, T))
    with torch.no_grad():
        want = ref(ids).logits

    # streaming path, using the reference model's own layers as the "checkpoint"
    cos, sin = build_rope_cache(T, cfg, "cpu", dtype=torch.float32)
    mask = build_attn_mask(T, "cpu", cfg.sliding_window)
    assert not torch.equal(mask, build_attn_mask(T, "cpu", None)), "sliding window not applied"
    assert int(mask[T - 1].sum()) == cfg.sliding_window

    with torch.no_grad():
        h = F.embedding(ids, ref.model.embed_tokens.weight)
        for i in range(cfg.n_layers):
            h, attn_w = decoder_layer_forward(ref.model.layers[i], h, cfg, cos, sin, mask,
                                              return_attn=True)
            assert attn_w.shape == (1, cfg.num_heads, T, T)   # post-GQA-expansion: 32 heads, not 8
        got = rms_norm(h, ref.model.norm.weight, cfg.rms_norm_eps) @ ref.lm_head.weight.T

    rel = ((got - want).norm() / want.norm()).item()
    if verbose:
        print(f"  streaming vs. MistralForCausalLM.forward: relative error {rel:.3e}")
    assert rel < 1e-5, (
        f"streaming forward disagrees with the real model (rel err {rel:.3e}). Check the weight "
        f"transpose, RoPE, repeat_kv, the mask and the RMSNorm cast order.")

    # Prove the check is not vacuous: disable RoPE (cos=1, sin=0 makes apply_rotary_pos_emb the
    # identity) and confirm the attention maps move. Compared on the maps rather than the logits
    # because a random-init MLP stack dilutes the difference by the time it reaches the head.
    with torch.no_grad():
        h0 = F.embedding(ids, ref.model.embed_tokens.weight)
        _, attn_ok = decoder_layer_forward(ref.model.layers[0], h0, cfg, cos, sin, mask,
                                           return_attn=True)
        _, attn_no = decoder_layer_forward(ref.model.layers[0], h0, cfg,
                                           torch.ones_like(cos), torch.zeros_like(sin), mask,
                                           return_attn=True)
    rel_broken = ((attn_no - attn_ok).norm() / attn_ok.norm()).item()
    assert rel_broken > 1e-2, "disabling RoPE changed nothing -- this check is vacuous!"
    if verbose:
        print(f"  RoPE is load-bearing (disabling it moves the attention maps by "
              f"{rel_broken:.3e})")

    # the STE grid must be a no-op on GPTQ's own output (the section-13 warm-start guarantee)
    W0 = torch.randn(64, 64) * 0.05
    Xd = torch.randn(2000, 64)
    Hd = (2.0 * Xd.T @ Xd / Xd.shape[0]).double()
    Wq, sc = gptq_quantize_layer(W0.clone(), Hd, bits=4, group_size=16, act_order=True,
                                 return_scale=True, work_dtype=torch.float64)
    qmax = 7
    rel_ste = ((torch.clamp(torch.round(Wq / sc), -qmax, qmax) * sc - Wq).norm() / Wq.norm()).item()
    assert rel_ste < 1e-6, f"return_scale is not the grid GPTQ used (rel err {rel_ste:.3e})"
    if verbose:
        print(f"  GPTQ return_scale is exact (re-quantization relative error {rel_ste:.3e})")

    free(ref)
    return rel


print("Validation 7a: streaming forward vs. real MistralForCausalLM (tiny random model)...")
_validate_streaming_against_hf()
print("PASSED -- the streaming forward reproduces the reference implementation.\n")

## 8. The pipeline passes

Everything above is machinery; these two functions are the pipeline.

**`pass_score`** walks the 32 layers with all-float weights and returns, per layer, the
JAB-Hessian trace and the HAWQ-V2 sensitivity table. Scoring must complete before the allocator
can choose bit-widths, so the adaptive experiment is genuinely two streaming passes (score, then
apply) -- there is no way to fuse them.

**`pass_quantize_eval`** walks the layers quantizing as it goes, propagating both a calibration
cache and an evaluation cache, and scores perplexity at the end. It covers every experiment via
`bits_fn`: `None` -> the fp16 control; `lambda i: 4` -> uniform; `assignment` -> adaptive;
`finetune=True` -> Objective 1.

### How the joint fine-tune keeps its float reference without a second model

The GPT-2 version needed `model_float` resident alongside `model_joint`, because `target_A` must
come from untouched full-precision weights -- distilling a quantized model toward its own
degraded output would defeat the objective. Two 7B copies is 29 GB.

Streaming reproduces those semantics with **two hidden-state caches** (268 MB total) plus a
100 MB fp32 snapshot of the current layer's float Q/K/V. Per layer:

1. snapshot the layer's float Q/K/V, and propagate `h_f` through the layer while it is still
   float -- so `h_f` always carries activations from an all-float stack, exactly what
   `model_float` provided;
2. GPTQ-quantize Q/K/V, capturing GPTQ's own per-(channel, group) scale;
3. fine-tune with `X = input_layernorm(h_q)` (quantized upstream, matching `model_joint`) and
   `target_A, target_attn = compute_attention(float snapshot, input_layernorm(h_f))`;
4. propagate `h_q` through the now-quantized layer.

`input_layernorm` is shared by both paths because norms are never quantized. Step 1 before step 2
is what makes it work -- once the module is quantized its float weights are gone, so the snapshot
and the `h_f` propagation must happen first.

The rest of the fine-tune logic is carried over unchanged, including the fix for the
scale-mismatch bug found on the GPT-2 version: `STEQuantize`'s grid must be **GPTQ's own**
per-(output channel, group) scale. The original derived a fresh per-input-row scale, which
re-quantized the optimized GPTQ solution onto a coarser, misaligned grid and injected ~9-10%
relative weight error *before Adam's first step* -- validated as exact in section 7. Also kept:
best-iterate tracking, and the safety net that commits fine-tuned weights only if they beat the
GPTQ-only starting loss, guaranteeing per layer that fine-tuning cannot worsen the local
reconstruction objective.

In [ ]:
class STEQuantize(torch.autograd.Function):
    """
    Straight-through estimator for fake quantization: rounds onto the bit-grid in the forward pass
    (so the loss sees quantization error) but passes the gradient through unchanged, so Adam can
    move the underlying float weights to compensate.
    """

    @staticmethod
    def forward(ctx, w, scale, bits):
        qmax = 2 ** (bits - 1) - 1
        return torch.clamp(torch.round(w / scale), -qmax, qmax) * scale

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output, None, None


def ste_quantize(w, scale, bits):
    return STEQuantize.apply(w, scale, bits)


def pass_score(reader, config, cfg, calib_ids, *, samples=None, n_score_batches=None,
               n_hessian_batches=None, lambda_kl=None, group_size=None, bit_widths=None,
               perturb_mode=None, verbose=True):
    """
    Streaming scoring pass over an all-float model.

    Returns (jab_scores, allocator_input):
      jab_scores      -- {layer_name: Hessian trace of the attention-aware loss}
      allocator_input -- HAWQ-V2 style {layer_name: {bits: trace * ||Q(W)-W||_F^2}}
    """
    samples = HUTCH_SAMPLES if samples is None else samples
    n_score_batches = JAB_N_BATCHES if n_score_batches is None else n_score_batches
    n_hessian_batches = HESSIAN_N_BATCHES if n_hessian_batches is None else n_hessian_batches
    lambda_kl = LAMBDA_KL if lambda_kl is None else lambda_kl
    group_size = GROUP_SIZE if group_size is None else group_size
    bit_widths = BIT_WIDTHS if bit_widths is None else bit_widths
    perturb_mode = PERTURB_MODE if perturb_mode is None else perturb_mode

    reset_vram_peak()
    h = embed_cache(reader, calib_ids)
    T = h[0].shape[1]
    cos16, sin16 = build_rope_cache(T, cfg, DEVICE, dtype=GPU_DTYPE)
    cos32, sin32 = build_rope_cache(T, cfg, DEVICE, dtype=torch.float32)
    mask = build_attn_mask(T, DEVICE, cfg.sliding_window)

    jab_scores, allocator_input = {}, {}
    print(f"Scoring {cfg.n_layers} layers: {samples} Hutchinson probes x {n_score_batches} "
          f"batches, loss = MSE + {lambda_kl}*KL, perturbation mode '{perturb_mode}'")

    for i in range(cfg.n_layers):
        layer = load_decoder_layer(reader, config, i)
        name = layer_name(i)

        # --- Hessian H = 2 X^T X (shared by q/k/v) ---
        H = hessian_from_hidden(layer, h[:n_hessian_batches], cfg.hidden_size, DEVICE)

        # --- JAB-Hessian trace of the attention-aware loss ---
        # fp32 throughout: double-backward through a fp16 softmax is numerically useless.
        W32 = qkv_weights_io(layer, dtype=torch.float32)
        w_flat = flatten_weights(*W32).clone().requires_grad_(True)
        traces = []
        for b in h[:n_score_batches]:
            X = layer.input_layernorm(b).to(torch.float32)
            with torch.no_grad():
                # Targets come from this layer's own float weights, so the loss is exactly 0 at
                # w = w_true and the trace is the curvature of the reconstruction loss there.
                tA, tattn = compute_attention(*W32, X, cfg, cos=cos32, sin=sin32, attn_mask=mask)
            tattn = tattn if lambda_kl else None

            def loss_fn(p):
                return attention_loss(p, X, tA, cfg, target_attn=tattn, lambda_kl=lambda_kl,
                                      cos=cos32, sin=sin32, attn_mask=mask)

            traces.append(hutchinson_trace_estimator(loss_fn, w_flat, samples=samples).item())
            free(X, tA, tattn)
        trace = sum(traces) / len(traces)
        jab_scores[name] = trace
        free(w_flat)

        # --- HAWQ-V2 table: Omega_i(bits) = trace_i * ||Q(W_i) - W_i||_F^2, summed over q/k/v ---
        table = {}
        for bits in bit_widths:
            pert = sum(grid_perturbation(W, bits, group_size=group_size, H=H, mode=perturb_mode)
                       for W in W32)
            table[bits] = trace * pert
        allocator_input[name] = table
        del H, W32                    # del, not free(...) -- see free()'s docstring
        free()

        if verbose:
            level = "HIGH" if trace > 100 else "MEDIUM" if trace > 10 else "LOW"
            print(f"  {name}: trace={trace:.4f} [{level}]  " +
                  ", ".join(f"{b}b={v:.3e}" for b, v in table.items()))

        # --- propagate the float cache to the next layer, then release the layer ---
        with torch.no_grad():
            for j in range(len(h)):
                h[j] = decoder_layer_forward(layer, h[j], cfg, cos16, sin16, mask)[0]
        del layer                     # MUST be `del`: free(layer) would leave `layer` bound here,
        free()                        # and the next load would allocate while it is still live

    free(h, cos16, sin16, cos32, sin32, mask)
    vram("after scoring pass")
    return jab_scores, allocator_input


def pass_quantize_eval(reader, config, cfg, calib_ids, eval_windows, *, bits_fn=None,
                       finetune=False, n_hessian_batches=None, group_size=None, lambda_kl=None,
                       steps_per_block=None, lr=None, grad_clip_norm=None, layer_indices=None,
                       label="", verbose=True):
    """
    Streaming quantize-and-evaluate pass.

    bits_fn: None -> leave weights in fp16 (the control); else layer_idx -> bit-width.
    finetune: additionally run the STE joint fine-tune of Objective 1 on each layer, against a
        float reference carried in a parallel hidden cache (see the markdown above).
    layer_indices: restrict fine-tuning to these layers; the rest still get the GPTQ warm start.

    Returns perplexity over `eval_windows`.
    """
    n_hessian_batches = HESSIAN_N_BATCHES if n_hessian_batches is None else n_hessian_batches
    group_size = GROUP_SIZE if group_size is None else group_size
    lambda_kl = LAMBDA_KL if lambda_kl is None else lambda_kl
    steps_per_block = JOINT_STEPS_PER_BLOCK if steps_per_block is None else steps_per_block
    lr = JOINT_LR if lr is None else lr
    grad_clip_norm = JOINT_GRAD_CLIP if grad_clip_norm is None else grad_clip_norm
    tune = None if layer_indices is None else set(layer_indices)

    print(f"\n=== streaming pass: {label} ===")
    reset_vram_peak()
    h_q = embed_cache(reader, calib_ids)
    h_f = [t.clone() for t in h_q] if finetune else None
    e_h = embed_cache(reader, [w[0] for w in eval_windows])

    Tc = h_q[0].shape[1]
    cos_c, sin_c = build_rope_cache(Tc, cfg, DEVICE, dtype=GPU_DTYPE)
    cos_c32, sin_c32 = build_rope_cache(Tc, cfg, DEVICE, dtype=torch.float32)
    mask_c = build_attn_mask(Tc, DEVICE, cfg.sliding_window)

    Te = e_h[0].shape[1]
    cos_e, sin_e = build_rope_cache(Te, cfg, DEVICE, dtype=GPU_DTYPE)
    mask_e = build_attn_mask(Te, DEVICE, cfg.sliding_window)

    for i in range(cfg.n_layers):
        layer = load_decoder_layer(reader, config, i)
        note = ""

        # (1) float snapshot + float-cache propagation, BEFORE the module is quantized
        W_float32 = None
        if finetune:
            W_float32 = tuple(W.clone() for W in qkv_weights_io(layer, dtype=torch.float32))
            with torch.no_grad():
                for j in range(len(h_f)):
                    h_f[j] = decoder_layer_forward(layer, h_f[j], cfg, cos_c, sin_c, mask_c)[0]

        if bits_fn is not None:
            bits = bits_fn(i)
            # (2) GPTQ warm start. Hessian comes from h_q, so it reflects layers 0..i-1 already
            #     being quantized -- the sequential GPTQ behaviour the GPT-2 notebook had.
            H = hessian_from_hidden(layer, h_q[:n_hessian_batches], cfg.hidden_size, DEVICE)
            want_scales = finetune and (tune is None or i in tune)
            out = quantize_qkv(qkv_of(layer), H, bits, group_size=group_size,
                               return_scales=want_scales)
            free(H)
            note = f"{bits} bits"

            # (3) STE joint fine-tune against the float reference
            if want_scales:
                ws, scales = out
                # w_flat is seeded from GPTQ's fp32 output (not the fp16 stored weight) and
                # scale_flat is GPTQ's own grid, so ste_quantize is a no-op at step 0.
                w_flat = flatten_weights(*ws).clone().requires_grad_(True)
                scale_flat = flatten_weights(*scales).to(w_flat.dtype)
                free(ws, scales)
                opt = torch.optim.Adam([w_flat], lr=lr)

                # rotate through the calibration set rather than every layer reusing batch 0..n
                start = (i * steps_per_block) % len(h_q)
                order = [(start + s) % len(h_q) for s in range(steps_per_block)]

                def targets(j):
                    with torch.no_grad():
                        X = layer.input_layernorm(h_q[j]).to(torch.float32)
                        Xf = layer.input_layernorm(h_f[j]).to(torch.float32)
                        tA, tattn = compute_attention(*W_float32, Xf, cfg, cos=cos_c32,
                                                      sin=sin_c32, attn_mask=mask_c)
                    return X, tA, (tattn if lambda_kl else None), Xf

                with torch.no_grad():
                    init = []
                    for j in order:
                        X, tA, tattn, Xf = targets(j)
                        init.append(attention_loss(w_flat.detach(), X, tA, cfg, target_attn=tattn,
                                                   lambda_kl=lambda_kl, cos=cos_c32, sin=sin_c32,
                                                   attn_mask=mask_c).item())
                        free(X, tA, tattn, Xf)
                    init_loss = sum(init) / len(init)

                best_loss, best_w = init_loss, w_flat.detach().clone()
                for j in order:
                    X, tA, tattn, Xf = targets(j)
                    opt.zero_grad()
                    loss = attention_loss(ste_quantize(w_flat, scale_flat, bits), X, tA, cfg,
                                          target_attn=tattn, lambda_kl=lambda_kl,
                                          cos=cos_c32, sin=sin_c32, attn_mask=mask_c)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_([w_flat], grad_clip_norm)
                    opt.step()
                    if loss.item() < best_loss:
                        best_loss = loss.item()
                        with torch.no_grad():
                            best_w = ste_quantize(w_flat, scale_flat, bits).detach().clone()
                    free(X, tA, tattn, Xf, loss)

                # Safety net: commit only if fine-tuning beat the GPTQ-only starting point.
                # Otherwise the module already holds that GPTQ-only solution -- leave it alone.
                if best_loss <= init_loss:
                    write_qkv_flat(layer, best_w, cfg)
                    note += f", FT {init_loss:.6f} -> {best_loss:.6f}"
                else:
                    note += f", kept GPTQ-only ({init_loss:.6f} vs FT {best_loss:.6f})"
                del w_flat, scale_flat, best_w, opt, targets
                free()
            elif finetune:
                note += ", GPTQ-only (not in layer_indices)"
        del W_float32
        free()

        # (4) propagate the quantized caches, then release the layer
        with torch.no_grad():
            for j in range(len(h_q)):
                h_q[j] = decoder_layer_forward(layer, h_q[j], cfg, cos_c, sin_c, mask_c)[0]
            for j in range(len(e_h)):
                e_h[j] = decoder_layer_forward(layer, e_h[j], cfg, cos_e, sin_e, mask_e)[0]
        del layer                     # see the note in pass_score -- `del`, not free(layer)
        free()

        if verbose:
            print(f"  layer {i:>2}/{cfg.n_layers}: {note or 'fp16 (no quantization)'}")

    free(h_q, h_f, cos_c, sin_c, cos_c32, sin_c32, mask_c)
    ppl = score_perplexity(reader, e_h, eval_windows, cfg)
    free(e_h, cos_e, sin_e, mask_e)
    vram(f"after '{label}'")
    print(f"=== {label}: perplexity {ppl:.3f} ===")
    return ppl

## 9. Setup

Downloads the checkpoint (~14.5 GB, a few minutes) and builds the shared calibration and
evaluation sets. Every experiment below reuses these, so all comparisons are apples-to-apples.

`mistralai/Mistral-7B-v0.1` is a **gated** repo: accept the license on the Hub and
`huggingface-cli login` (or set the `HF_TOKEN` secret in Colab) first, or this cell will 401.

In [ ]:
print("Fetching the checkpoint index and shards (cached after the first run)...")
reader = CheckpointReader(MODEL_ID, DEVICE)
config = AutoConfig.from_pretrained(MODEL_ID)
config._attn_implementation = "eager"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

ATTN = AttnConfig(config)
print(ATTN)
print(f"Q/K/V parameters per layer: {ATTN.qkv_numel:,}")
print(f"checkpoint dtype: {config.torch_dtype if hasattr(config, 'torch_dtype') else 'n/a'}"
      f"  ->  layer compute dtype: {GPU_DTYPE}")

print("\nBuilding calibration and evaluation sets...")
calib_ids = build_calibration_ids(tokenizer)
eval_windows = build_eval_windows(tokenizer)
scored_tokens = sum(min(t, w.shape[1] - 1) for w, t in eval_windows)
print(f"{len(calib_ids)} calibration chunks of {CALIB_SEQ_LEN} tokens")
print(f"{len(eval_windows)} evaluation windows of {EVAL_MAX_LENGTH} tokens "
      f"(~{scored_tokens:,} scored tokens)")
print(f"hidden-state caches: calib {len(calib_ids) * CALIB_SEQ_LEN * ATTN.hidden_size * 2 / 1e6:.0f} MB, "
      f"eval {len(eval_windows) * EVAL_MAX_LENGTH * ATTN.hidden_size * 2 / 1e6:.0f} MB")

results = {}

## 10. fp16 control (validation 7b)

The unquantized streaming perplexity. This is not one of the experiments -- it is the check that
the streaming forward is right on the *real* model, at full scale, where section 7a's tiny-model
test cannot reach. Mistral-7B should land in the ~5-6 range on WikiText-2; a wildly different
number means something in the streaming path is wrong and nothing below is trustworthy.

In [ ]:
if RUN_FP16_BASELINE:
    results["fp16"] = pass_quantize_eval(reader, config, ATTN, calib_ids, eval_windows,
                                         bits_fn=None, label="fp16 control (no quantization)")
    if not (3.0 < results["fp16"] < 12.0):
        print(f"\nWARNING: fp16 perplexity {results['fp16']:.3f} is outside the expected ~5-6 "
              f"range for Mistral-7B on WikiText-2. Investigate the streaming forward before "
              f"reading anything into the quantization results below.")
else:
    print("RUN_FP16_BASELINE=False -- skipping the control pass.")

## 11. Uniform 4-bit GPTQ baseline
Every layer's `q_proj`/`k_proj`/`v_proj` at a flat 4 bits -- the comparison point for adaptive
allocation. Hessians are collected from the propagating calibration cache, so layer *i*'s
Hessian already reflects layers `0..i-1` being quantized: the sequential GPTQ behaviour the
GPT-2 notebook had.

In [ ]:
results["uniform4"] = pass_quantize_eval(reader, config, ATTN, calib_ids, eval_windows,
                                         bits_fn=lambda i: 4, label="uniform 4-bit GPTQ")

## 12. JAB-Hessian adaptive allocation

Pass one scores every layer; the allocator picks bit-widths under an average-bit budget; pass two
applies them and evaluates.

**Budget note (carried over from the GPT-2 notebook, and it still applies).** We use
`target_avg_bits = 4.3`, not `4.0`. At exactly 4.0 the budget lands on a `BIT_WIDTHS` grid point:
below 4 bits the HAWQ-V2 perturbation term climbs steeply, so every layer's cheap 16->8 and 8->4
downgrades are exhausted first regardless of its trace, and the loop halts the instant all layers
hit the 4-bit floor -- before any layer competes for a downgrade below or an upgrade above it.
That is a budget sitting on a cliff, not a broken allocator. The sweep below demonstrates it: the
allocator differentiates cleanly at non-grid-aligned budgets, and the sweep is free because it
reuses the scores already computed.

In [ ]:
torch.manual_seed(42)
jab_scores, allocator_input = pass_score(reader, config, ATTN, calib_ids)

TARGET_AVG_BITS = 4.3
budget = TARGET_AVG_BITS * len(allocator_input)
assignment, cost_used, sensitivity = greedy_allocate(allocator_input, budget)
print(f"\nBudget {budget:.1f} bits ({TARGET_AVG_BITS} avg x {len(allocator_input)} layers)")
print(f"Allocated: {cost_used:.1f} bits total, {cost_used / len(allocator_input):.2f} avg, "
      f"total sensitivity {sensitivity:.4e}")
print("  " + ", ".join(f"L{layer_idx_of(n)}={b}" for n, b in assignment.items()))

print("\nBudget sensitivity sweep (free -- reuses the scores above):")
for avg_bits in [3.5, 4.0, 4.3, 4.5, 6.0]:
    alloc, c, _ = greedy_allocate(allocator_input, avg_bits * len(allocator_input))
    print(f"  avg_bits={avg_bits}: cost {c:.1f}, distinct widths {sorted(set(alloc.values()))}")

In [ ]:
results["adaptive"] = pass_quantize_eval(
    reader, config, ATTN, calib_ids, eval_windows,
    bits_fn=lambda i: assignment[layer_name(i)], label=f"JAB adaptive ({TARGET_AVG_BITS} avg bits)")

## 13. Joint attention-aware fine-tuning (Objective 1)

Sections 11-12 used `attention_loss` only as an importance **score** -- to decide *how many bits*
each layer gets. The weights themselves were still produced by GPTQ minimizing the conventional
`||XW - XW_hat||^2`, not the attention-output loss Objective 1 asks for.

This closes that gap: each layer's GPTQ-initialized `W_Q, W_K, W_V` are wrapped in a
straight-through quantizer and fine-tuned with Adam to directly minimize
`min_{W_hat} L(A(X), A_hat(X))`. See section 8's markdown for how the float reference is carried
in a parallel hidden cache instead of a second 14.5 GB model.

In [ ]:
if RUN_JOINT_FINETUNE:
    results["joint"] = pass_quantize_eval(
        reader, config, ATTN, calib_ids, eval_windows,
        bits_fn=lambda i: 4, finetune=True, layer_indices=JOINT_LAYERS,
        label="joint attention-aware fine-tuned (4-bit)")
else:
    print("RUN_JOINT_FINETUNE=False -- skipping Objective 1.")

## 14. Compare results

In [ ]:
LABELS = {"fp16": "fp16 control (unquantized)",
          "uniform4": "Uniform 4-bit GPTQ",
          "adaptive": f"Adaptive JAB-Hessian ({TARGET_AVG_BITS} avg bits)",
          "joint": "Joint attention-aware fine-tuned (4-bit)"}

print(f"{MODEL_ID}  ({ATTN.n_layers} layers, hidden {ATTN.hidden_size}, "
      f"{ATTN.num_heads}q/{ATTN.num_kv_heads}kv heads)")
print(f"WikiText-2 subset perplexity over {scored_tokens:,} scored tokens "
      f"({len(eval_windows)} windows of {EVAL_MAX_LENGTH}); identical windows for every row.\n")
for key, label in LABELS.items():
    if key in results:
        print(f"  {label:<48}: {results[key]:.3f}")

base = results.get("uniform4")
if base is not None:
    print()
    for key in ("adaptive", "joint"):
        if key in results:
            print(f"  {LABELS[key]:<48}: {results[key] - base:+.3f} vs. uniform 4-bit")

reader.close()
free(reader)